# ζ(2): carries, six prime shells, and integer normalization

This notebook supports **Lemma 6.20 (Integer normalization, `prop:zeta2-int`)**, the arithmetic part of **Lemma 6.21 (`lem:z2-growth`)**, and Appendix `app:z2-blocks`.

It runs independently of notebook 01. Exact finite checks verify the implemented formulas and shell identities; the exhaustive floor-arrangement certificate covers all real points in the indicated chambers. The passage to uniform divisibility and the prime-number-theorem limit is the paper's mathematical argument. No Lean code is written or executed.

In [1]:
from pathlib import Path
import sys
# Works from this notebook folder or from the unzipped bundle root.
HERE = next((p.resolve() for p in [Path.cwd(), Path.cwd()/"zeta2"]
             if (p/"companion.py").is_file()), None)
if HERE is None:
    raise FileNotFoundError("Run this notebook from the zeta2 folder or bundle root.")
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE/"certificates"))
from fractions import Fraction as Q
from IPython.display import display, Markdown
import companion as z2
print("Local Python support loaded. No other notebook needs to be run.")

Local Python support loaded. No other notebook needs to be run.


## 1. Notation and the normalization to be checked

Let $d_m=\operatorname{lcm}(1,\ldots,m)$, $d_0=1$, and
\[
C_n=\prod_{u=9,10,11}\binom{un}{8n},\qquad
\mathcal D_n=\mathcal D_n^{\rm car}\mathcal D_n^{\rm sh}.
\]
Lemma 6.20 states
\[
(\widetilde p_n,\widetilde q_n)=
\frac{C_nd_{12n}d_{13n}}{\mathcal D_n}(p_n,q_n)\in\mathbb Z^2.
\]
These are integer **representatives**, not necessarily the coprime numerator and denominator. The true reduced denominator is at most $|\widetilde q_n|$.

Use $\mathcal D_n^{\rm car}$ and $\mathcal D_n^{\rm sh}$ throughout, and $\mathcal D_n$ for their product. The symbol $G_n$ is reserved for the Meijer $G$ function.

## 2. Carry valuations from Legendre's formula

For primes $p^2>50n$, the factorials in the relevant binomial terms have only the first Legendre contribution. With $x=n/p$, $y=j/p$, the lower bound is
\[
\begin{aligned}
\rho(x,y)={}&\lfloor11x\rfloor-\lfloor y\rfloor-\lfloor11x-y\rfloor\\
&+\sum_{(u,v)=(10,2),(9,1),(11,3)}
\bigl(\lfloor ux+y\rfloor-\lfloor8x\rfloor-\lfloor vx+y\rfloor\bigr),
\qquad \phi(x)=\min_{0\le y<1}\rho(x,y).
\end{aligned}
\]
Thus
\[
\mathcal D_n^{\rm car}=\prod_{\sqrt{50n}<p\le13n}p^{\phi(n/p)}.
\]
The implementation uses exact rational breakpoints $\{cx\}$ with $c\in\{11,-10,-9,-11,-2,-1,-3\}$ and includes both boundaries and interior cells in the minimum. It does not optimize over a numerical grid.

In [2]:
from rate_certificate import rho,phi
from math import comb

def valuation(integer,prime):
    assert integer != 0
    result=0
    while integer%prime == 0:
        integer//=prime
        result+=1
    return result

checked_terms=0
for n in range(1,9):
    for p in z2.primes_to(13*n):
        if p*p <= 50*n:
            continue
        for j in range(11*n+1):
            term=comb(11*n,j)
            for u in (9,10,11):
                term*=comb(j+u*n,8*n)
            carry=rho(Q(n,p),Q(j,p))
            assert valuation(term,p) == carry
            assert carry >= phi(Q(n,p)) >= 0
            checked_terms+=1
print(f"Legendre/carry equality checked for {checked_terms} exact binomial terms.")

Legendre/carry equality checked for 5409 exact binomial terms.


## 3. Six additional shells

For each eligible prime in any of the following **open** intervals there is one further factor of $p$:
\[
(13n/2,8n),\ (13n/3,9n/2),\ (13n/4,10n/3),\
(13n/5,8n/3),\ (13n/6,9n/4),\ (13n/12,9n/8).
\]
Their product is $\mathcal D_n^{\rm sh}$. The total interval length divided by $n$ is exactly $233/120$. Strict endpoints and $p^2>50n$ are enforced in `common_divisors`.

Why an extra factor occurs: at minimum carry, the normalized local expression becomes a polynomial over $\mathbb F_p$ of degree less than $p-1$. Its complete residue sum vanishes: $\sum_{x\in\mathbb F_p}P(x)=0$ whenever $\deg P<p-1$. For the constant term this is $p=0$ in $\mathbb F_p$; for a monomial of degree $1\le d<p-1$, choose $a\in\mathbb F_p^\times$ with $a^d\ne1$ and rescale the sum by $x\mapsto ax$. Linearity gives the polynomial statement. The eight rows below describe the six shells after two internal splits.

In [3]:
shell_rate=sum((right-left for left,right in z2.SHELLS),Q(0))
assert shell_rate == Q(233,120)
print("shell rate =",shell_rate,"=",float(shell_rate))
chambers,seconds=z2.run_certificate("chamber_certificate.py")
assert chambers["status"] == "exact_chamber_certificate_passed"
assert chambers["total_y_cell_checks"] == 390
header="| x=n/p interval | floor(8x) | floor(11x) | minimum carry | e | upper digits λ | low carries δ |"
table=[header,"|---|---:|---:|---:|---:|---|---|"]
for row in chambers["regions"]:
    a,M,h,e,lam,delta=row["signature"]
    lo,hi=row["x_interval"]
    table.append(f"| ({lo}, {hi}) | {a} | {M} | {h} | {e} | {lam} | {delta} |")
display(Markdown("\n".join(table)))
print(f"Exhaustive arrangement: 390 cells/boundaries checked in {seconds}s.")

shell rate = 233/120 = 1.9416666666666667


| x=n/p interval | floor(8x) | floor(11x) | minimum carry | e | upper digits λ | low carries δ |
|---|---:|---:|---:|---:|---|---|
| (1/8, 2/13) | 1 | 1 | 0 | 0 | [1, 1, 1] | [0, 0, 0] |
| (2/9, 3/13) | 1 | 2 | 2 | 0 | [2, 2, 2] | [1, 1, 0] |
| (3/10, 4/13) | 2 | 3 | 1 | 0 | [2, 3, 3] | [0, 1, 0] |
| (3/8, 5/13) | 3 | 4 | 0 | 0 | [3, 3, 4] | [0, 0, 0] |
| (4/9, 5/11) | 3 | 4 | 1 | 0 | [4, 5, 5] | [0, 1, 0] |
| (5/11, 6/13) | 3 | 5 | 2 | 1 | [4, 5, 5] | [0, 1, 0] |
| (8/9, 10/11) | 7 | 9 | 0 | 0 | [8, 9, 10] | [0, 0, 0] |
| (10/11, 12/13) | 7 | 10 | 1 | 1 | [8, 9, 10] | [0, 0, 0] |

Exhaustive arrangement: 390 cells/boundaries checked in 0.024s.


The exhaustive scope is finite for a mathematical reason: the order of the boundaries $\{cx\}$ changes only when $cx$ or $(c-c')x$ is an integer. The certificate enumerates all these rational cuts and each intervening chamber, including internal boundary points. The exceptional split values $x=5/11,10/11$ cannot occur for the eligible primes in those shells; the divisibility explanation is recorded in the source certificate.

The next cell **reruns**, rather than loads, the four supplied congruence diagnostics. They cover all eight subregions and use an independent standard-library implementation for selected finite sums. These diagnostics are complementary to, and do not replace, the uniform argument.

In [4]:
diagnostics=[]
for script in ("check_local_kernel.py", "check_main_shell.py",
               "check_fifth_shell.py", "check_shell_independent.py"):
    result,seconds=z2.run_certificate(script)
    # A nonzero exit or failed assertion raises before a row can be recorded.
    diagnostics.append((script,"all script assertions passed",seconds))
diagnostics

[('check_local_kernel.py', 'all script assertions passed', 0.805),
 ('check_main_shell.py', 'all script assertions passed', 3.093),
 ('check_fifth_shell.py', 'all script assertions passed', 8.513),
 ('check_shell_independent.py', 'all script assertions passed', 0.23)]

## 4. Check Lemma 6.20 on actual approximants

The following computes $(p_n,q_n)$ from the CMF matrix itself, constructs the two divisors independently from the prime definitions, and checks integer denominators exactly. It also verifies that normalization preserves the ratio. Finite $n$ checks are regression tests of the formulas; they are not the all-$n$ proof.

In [5]:
table=["| n | digits in carry divisor | shell divisor | digits in q̃_n | integral pair |",
       "|---:|---:|---:|---:|---|"]
for n in range(1,13):
    p_n,q_n=z2.matrix_readout(n)
    carry,shell=z2.common_divisors(n)
    scale=z2.normalization_C(n)*z2.lcm_to(12*n)*z2.lcm_to(13*n)
    p_tilde=p_n*scale/(carry*shell)
    q_tilde=q_n*scale/(carry*shell)
    assert p_tilde.denominator == q_tilde.denominator == 1
    assert p_tilde/q_tilde == p_n/q_n
    table.append(f"| {n} | {len(str(carry))} | {shell} | {len(str(abs(int(q_tilde))))} | yes |")
display(Markdown("\n".join(table)))

| n | digits in carry divisor | shell divisor | digits in q̃_n | integral pair |
|---:|---:|---:|---:|---|
| 1 | 2 | 1 | 24 | yes |
| 2 | 8 | 1 | 44 | yes |
| 3 | 7 | 23 | 69 | yes |
| 4 | 12 | 899 | 92 | yes |
| 5 | 19 | 37 | 112 | yes |
| 6 | 17 | 82861 | 139 | yes |
| 7 | 31 | 1776083 | 152 | yes |
| 8 | 32 | 190747 | 177 | yes |
| 9 | 36 | 17120443 | 202 | yes |
| 10 | 42 | 27433619 | 221 | yes |
| 11 | 44 | 13881169 | 248 | yes |
| 12 | 51 | 30929369 | 266 | yes |

## 5. Exact carry integral — arithmetic part of Lemma 6.21

The prime number theorem gives
\[
\gamma=\int_{1/13}^{\infty}\phi(x)x^{-2}\,dx,
\qquad
\lim_{n\to\infty}\frac1n\log\frac{d_{12n}d_{13n}}
{\mathcal D_n^{\rm car}\mathcal D_n^{\rm sh}}
=25-\gamma-\frac{233}{120}.
\]
The one-periodic profile $\phi$ is constant on 94 rational cells of $[0,1)$. For a cell $(a,b)$ of height $h$, its contribution on the $k$th period is $h[(k+a)^{-1}-(k+b)^{-1}]$. The program sums 2000 periods using directed integer rounding at scale $10^{15}$. If $m=\int_0^1\phi$, the omitted tail lies between $m/(K+1)$ and $m/(K-1)$. Endpoints have measure zero in this integral.

The source's legacy JSON field `status="NUMERICAL_CANDIDATE"` is preserved for provenance. The inequalities asserted here use **exact Fraction bounds** recomputed from the source; no decimal display or status string is evidence for them.

In [6]:
rate,seconds=z2.run_certificate("rate_certificate.py")
gamma_lower=Q(rate["gamma_lower"])
gamma_upper=Q(rate["gamma_upper"])
arithmetic_upper=25-gamma_lower-shell_rate
assert rate["cells"] == 94 and rate["periods"] == 2000
assert gamma_lower > Q("12.128427")
assert gamma_lower < gamma_upper
assert arithmetic_upper < Q("10.929906")
print("gamma enclosure:",float(gamma_lower),"< gamma <",float(gamma_upper))
print("25 − gamma − 233/120 <",float(arithmetic_upper))
print("All displayed threshold comparisons were asserted as exact fractions.")

gamma enclosure: 12.12842797479104 < gamma < 12.128428404323277
25 − gamma − 233/120 < 10.929905358542294
All displayed threshold comparisons were asserted as exact fractions.


**Arithmetic conclusion used by the paper:** after the integer normalization of Lemma 6.20, the remaining arithmetic multiplier has exponential rate less than $10.929906$. Combining this with the coefficient growth rate below $38.168733$ gives $h<49.098639$. Notebook 03 recomputes both certificates and the final measure comparison independently.